# Design / Outfit Transfer — MASK REVIEW (arquitetura generalizada)

Linha **experimental**, 100% **nao-generativa**. Nao altera Run 003, FLUX,
Flow 01, quality gates nem o pipeline oficial.

```
mask generator -> estrategias genericas -> overrides por personagem
   -> mascaras -> MASK REVIEW HUMANA -> warp/composicao (BLOQUEADO)
```

## Tres camadas

| camada | onde vive | exemplo |
|---|---|---|
| **GLOBAL** | `scripts/chibi/mask_engine.py` | "tecido lateral acima da cintura e manga" |
| **CHARACTER-SPECIFIC** | `characters/<id>/masks.yaml` | `leg_left: 0.26` da waifu_001 |
| **REGIONAL** | `mask_profile.DEFAULT_RIGIDITY` | ornamento e rigido; tecido e cloth |

Os limiares (luminancia, eixo cromatico, cintura, colunas das pernas) sao
**derivados da propria arte** — Otsu, percentis e perfil de silhueta. Nenhum
numero medido na waifu_001 e default global. Cada parametro reporta sua
**procedencia**: `derived:` ou `override:`.

Uma personagem **sem** `masks.yaml` roda apenas com a estrategia generica.

> Warp e composicao seguem **bloqueados** ate aprovacao humana das mascaras.


In [ ]:
#@title 1. Ambiente e repositorio { display-mode: "form" }
#@markdown Instala as dependencias (todas permissivas) e localiza o repo.
REPO_URL = "https://github.com/BloomRX/ChibiCreate"  #@param {type:"string"}
BRANCH   = "arena/01a07ece-chibicreate"  #@param {type:"string"}
ATUALIZAR_REPO = True  #@param {type:"boolean"}
#@markdown Mantenha marcado: garante que o runtime nao rode codigo antigo.

import subprocess, sys, os
from pathlib import Path

def sh(*a):
    print("$", " ".join(a))
    subprocess.run(a, check=False)

IN_COLAB = "google.colab" in sys.modules
# scikit-image >= 0.25 basta: o codigo detecta a API por versao
# (0.25.x e 0.26 divergem em remove_small_* e ThinPlateSplineTransform).
sh(sys.executable, "-m", "pip", "-q", "install",
   "numpy", "pillow", "scikit-image>=0.25", "scipy")

ROOT = None
for c in (Path.cwd(), *Path.cwd().parents):
    if (c / "scripts" / "chibi").is_dir():
        ROOT = c
        break
if ROOT is None and Path("ChibiCreate/scripts/chibi").is_dir():
    ROOT = Path("ChibiCreate").resolve()
if ROOT is None:
    sh("git", "clone", "--branch", BRANCH, REPO_URL, "ChibiCreate")
    ROOT = Path("ChibiCreate").resolve()
os.chdir(ROOT)

# SEMPRE atualizar: um clone antigo do runtime deixaria o notebook rodando
# codigo obsoleto e reproduzindo bugs ja corrigidos.
if ATUALIZAR_REPO:
    sh("git", "fetch", "--quiet", "origin", BRANCH)
    sh("git", "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}")

sys.path.insert(0, str(ROOT))

# limpar modulos ja importados, senao o Python reusa a versao velha da memoria
for _m in [m for m in sys.modules if m.startswith("scripts.chibi")]:
    del sys.modules[_m]

print("\nrepo:", ROOT)
subprocess.run(["git", "log", "--oneline", "-1"])

import numpy, PIL, skimage
print("numpy", numpy.__version__, "| pillow", PIL.__version__,
      "| scikit-image", skimage.__version__)
print("licencas: BSD-3-Clause / MIT-CMU / BSD-3-Clause  (todas comerciais)")


In [ ]:
#@title 2. Carregar Run 003 e full_body { display-mode: "form" }
#@markdown `run_003/output.png` **nao esta versionado** (`experiments/**/*.png`
#@markdown e gitignored). Faca upload dele. `full_body.png` vem do repo.
CHARACTER_ID = "waifu_001"  #@param {type:"string"}
RUN_003_PATH = ""  #@param {type:"string"}
#@markdown Deixe vazio para abrir o seletor de upload.

import sys
from pathlib import Path
import numpy as np
from PIL import Image

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
from scripts.chibi import design_transfer as dt

REF = ROOT / "characters" / CHARACTER_ID / "reference"
FULL_BODY = REF / "full_body.png"
assert FULL_BODY.exists(), f"nao encontrei {FULL_BODY}"

WORK = ROOT / "experiments" / "design_transfer" / "run_003"
(WORK / "masks").mkdir(parents=True, exist_ok=True)

base_path = Path(RUN_003_PATH) if RUN_003_PATH else None
if base_path is None or not base_path.exists():
    try:
        from google.colab import files
        print("Selecione run_003/output.png:")
        up = files.upload()
        name = next(iter(up))
        base_path = WORK / "run_003_input.png"
        base_path.write_bytes(up[name])
    except ImportError:
        raise SystemExit("Fora do Colab: preencha RUN_003_PATH.")

BASE = dt.load_rgba(base_path)
SOURCE = dt.load_rgba(FULL_BODY)

# VALIDACAO DE INTEGRIDADE — dois hashes distintos, por motivos distintos.
#   artifact_sha256 = bytes do arquivo. Muda se o PNG for reescrito, mesmo
#                     com pixels identicos.
#   pixel_sha256    = conteudo RGBA. Este e o que diz se a IMAGEM mudou.
import hashlib

def artifact_sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def pixel_sha256(img):
    return hashlib.sha256(np.array(img.convert("RGBA")).tobytes()).hexdigest()

RUN003_ARTIFACT_SHA = artifact_sha256(base_path)
RUN003_PIXEL_SHA = pixel_sha256(BASE)
SOURCE_ARTIFACT_SHA = artifact_sha256(FULL_BODY)

print("Run 003   :", base_path, BASE.size, BASE.mode)
print("  artifact_sha256:", RUN003_ARTIFACT_SHA)
print("  pixel_sha256   :", RUN003_PIXEL_SHA)
print("full_body :", FULL_BODY, SOURCE.size, SOURCE.mode)
print("  artifact_sha256:", SOURCE_ARTIFACT_SHA)

# Hash conhecido da arte-fonte desta personagem. E calibracao por personagem,
# nao regra global: personagem nova simplesmente nao tem entrada aqui.
SOURCE_SHA_ESPERADO = {
    "waifu_001":
        "2fdcd5f428f5980d63e31d4bf4a67aecbc11c1b101c19ca75f819db616cb8177",
}.get(CHARACTER_ID)

if SOURCE_SHA_ESPERADO is None:
    print("\n[nota] sem hash registrado para", CHARACTER_ID)
elif SOURCE_ARTIFACT_SHA != SOURCE_SHA_ESPERADO:
    raise SystemExit(
        "PARE: full_body.png diverge do esperado. A arte-fonte nao deve ser "
        "modificada.\n  esperado: " + SOURCE_SHA_ESPERADO +
        "\n  obtido  : " + SOURCE_ARTIFACT_SHA)
else:
    print("\nfull_body.png confere com o hash registrado.")

# A Run 003 nao tem hash fixo no repo (nao e versionada). Anote o valor
# acima: ele identifica exatamente qual imagem foi usada nesta sessao.
print("\n[HUMAN REVIEW REQUIRED] Anote o pixel_sha256 da Run 003 acima.")
print("Ele entra no recipe e identifica a imagem usada nesta sessao.")

if BASE.size != SOURCE.size:
    print("\n[nota] resolucoes diferentes — esperado. A REAL e a CHIBI tem")
    print("       geometrias distintas; e exatamente por isso que existe a")
    print("       etapa de correspondencia (9A-9E).")


In [ ]:
#@title 3. Ver as entradas (1 = Run 003, 2 = full_body) { display-mode: "form" }
import matplotlib.pyplot as plt

def flat(img, bg=(255, 255, 255)):
    c = Image.new("RGB", img.size, bg)
    c.paste(img, (0, 0), img)
    return c

fig, ax = plt.subplots(1, 2, figsize=(11, 6))
ax[0].imshow(flat(BASE));   ax[0].set_title("1. Run 003 (base preservada)")
ax[1].imshow(flat(SOURCE)); ax[1].set_title("2. full_body (fonte do design)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
#@title 4. GENERATE MASKS — generico + overrides { display-mode: "form" }
#@markdown Limiares sao DERIVADOS da propria arte. Um `characters/<id>/
#@markdown masks.yaml` pode sobrescrever pontos especificos — e opcional.
#@markdown Deixe `USAR_PERFIL` desmarcado para ver o comportamento 100% generico.
MIN_AREA = 120  #@param {type:"integer"}
USAR_PERFIL = True  #@param {type:"boolean"}

from scripts.chibi import mask_engine as me
from scripts.chibi import mask_profile as mp

REVIEW = me.review_masks(SOURCE,
                         character_id=CHARACTER_ID if USAR_PERFIL else None,
                         min_area=MIN_AREA)
MASKS = REVIEW.garment
PROTECTED = REVIEW.protected
BANNED = REVIEW.banned

print("PROCEDENCIA DOS PARAMETROS")
print(REVIEW.provenance_report())
print("\n  derivados da arte :", ", ".join(REVIEW.params.derived) or "-")
print("  vindos do perfil  :", ", ".join(REVIEW.params.overridden) or "(nenhum)")

total = int(np.count_nonzero(REVIEW.subject))
print("\nREGIOES PROTEGIDAS (a composicao nunca escreve aqui)")
for k in me.PROTECTED_REGIONS:
    n = int(np.count_nonzero(PROTECTED[k]))
    print(f"  {k:12}{n:8d}  {100*n/total:5.2f}%")

print("\nPECAS DE VESTUARIO")
print(f"  {'peca':16}{'pixels':>8}{'%subj':>8}{'overlap':>9}  estrategia")
for k in me.GARMENT_REGIONS:
    m = REVIEW.metrics[k]
    print(f"  {k:16}{m['pixels']:8d}{m['pct_subject']:8.2f}"
          f"{m['protected_overlap']:9d}  {m['strategy']}")

print("\nmask_protected_overlap_pixels == 0 em todas?",
      all(REVIEW.metrics[k]["protected_overlap"] == 0 for k in me.GARMENT_REGIONS))


In [ ]:
#@title 5. Cada mascara separadamente { display-mode: "form" }
fig, ax = plt.subplots(2, 3, figsize=(15, 11))
for a, name in zip(ax.ravel(), me.GARMENT_REGIONS):
    a.imshow(MASKS[name], cmap="gray")
    m = REVIEW.metrics[name]
    a.set_title(f"{name}\n{m['pixels']} px · {m['pct_subject']:.2f}%")
    a.axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 5, figsize=(18, 5))
for a, name in zip(ax, me.PROTECTED_REGIONS):
    a.imshow(PROTECTED[name], cmap="Reds")
    a.set_title(f"PROTEGIDA: {name}"); a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
#@title 6. Overlay de CADA mascara sobre full_body { display-mode: "form" }
#@markdown Olhe peca por peca: "essa mascara cobre exatamente a peca desejada?"
OVERLAY_ALPHA = 0.55  #@param {type:"slider", min:0.1, max:0.9, step:0.05}

fig, ax = plt.subplots(2, 3, figsize=(16, 13))
for a, name in zip(ax.ravel(), me.GARMENT_REGIONS):
    a.imshow(flat(me.overlay_single(SOURCE, MASKS[name],
                                    me.GARMENT_COLORS[name], OVERLAY_ALPHA)))
    a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
#@title 7. Overlay combinado + regioes protegidas { display-mode: "form" }
ov_all = me.overlay(SOURCE, MASKS, alpha=OVERLAY_ALPHA)
ov_prot = me.overlay(SOURCE, PROTECTED,
                     {k: (255, 0, 0) for k in PROTECTED}, 0.5)

fig, ax = plt.subplots(1, 3, figsize=(17, 7))
ax[0].imshow(flat(SOURCE));   ax[0].set_title("1. full_body original")
ax[1].imshow(flat(ov_all));   ax[1].set_title("todas as pecas")
ax[2].imshow(flat(ov_prot));  ax[2].set_title("regioes PROTEGIDAS (vermelho)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

MASK_PATHS = {}
for name, m in MASKS.items():
    p = WORK / "masks" / f"garment_{name}.png"
    Image.fromarray((m * 255).astype(np.uint8), "L").save(p)
    MASK_PATHS[name] = p
for name, m in PROTECTED.items():
    Image.fromarray((m * 255).astype(np.uint8), "L").save(
        WORK / "masks" / f"protected_{name}.png")
ov_all.save(WORK / "masks" / "overlay_all_on_fullbody.png")
ov_prot.save(WORK / "masks" / "overlay_protected.png")
print("mascaras salvas em", (WORK / "masks").relative_to(ROOT))


In [ ]:
#@title 8. [OBSOLETA] mapeamento por caixa do sujeito { display-mode: "form" }
#@markdown **Esta etapa foi desativada de proposito.**
#@markdown
#@markdown Ela mapeava a mascara da personagem REAL para a Run 003 usando so a
#@markdown caixa do sujeito. Isso e um **erro conceitual**: a mascara vive no
#@markdown espaco geometrico da REAL, e a Run 003 e CHIBI — outras proporcoes,
#@markdown outra escala, outras posicoes. Esticar uma caixa na outra nao faz a
#@markdown gola cair na gola nem a barra cair na barra.
#@markdown
#@markdown A correspondencia correta esta nas etapas 9A-9E abaixo:
#@markdown landmarks -> correspondencia -> transformacao -> mascara no espaco
#@markdown da CHIBI -> review.
print("Etapa 8 desativada. Ver CORRESPONDENCE REVIEW (9A-9E).")


## CORRESPONDENCE REVIEW — REAL <-> CHIBI

**Antes de qualquer warp ou composicao.**

A mascara da roupa foi definida sobre a personagem **REAL**. A Run 003 e
**CHIBI**. Aplicar uma na outra diretamente e invalido. Ordem obrigatoria:

```
landmarks -> correspondencia -> transformacao -> mascara em coords CHIBI
          -> review visual -> (so entao) composicao
```

**TPS e a transformacao, nunca a etapa de correspondencia.**

Correspondencia **parcial e normal**: so entram os landmarks presentes nas
duas imagens. O que existe so na REAL fica listado como diagnostico.

Detalhe desta arte: o contorno so sustenta 4 landmarks
(`top_of_head`, `silhouette_bottom`, `shoulder_left/right`). Pescoco, cintura,
quadril e tornozelos **nao** aparecem na silhueta — cabelo longo funde cabeca e
ombros, e a capa cobre o contorno lateral. Esses pontos vem de
`characters/<id>/landmarks.yaml`. Landmark chutado e pior que landmark ausente.


In [ ]:
#@title 9A. Landmarks da personagem REAL { display-mode: "form" }
#@markdown Derivados do contorno + overrides de `characters/<id>/landmarks.yaml`.
#@markdown Desmarque para ver so o que a deteccao automatica sustenta sozinha.
USAR_LANDMARKS_YAML = True  #@param {type:"boolean"}

import sys
for m in [k for k in list(sys.modules) if "character_correspondence" in k]:
    del sys.modules[m]
from scripts.chibi import character_correspondence as cc

OVR = cc.load_landmark_overrides(CHARACTER_ID) if USAR_LANDMARKS_YAML else {}

SRC_LM = cc.derive_landmarks(SOURCE)
derivados = list(SRC_LM.names())
if OVR.get("source"):
    SRC_LM = cc.apply_overrides(SRC_LM, OVR["source"], source="config")

print(f"derivados do contorno ({len(derivados)}): {derivados}")
print(f"total apos landmarks.yaml: {len(SRC_LM.names())}")
for n in SRC_LM.names():
    nx, ny = SRC_LM.normalized(n)
    print(f"  {n:18} norm({nx:.3f}, {ny:.3f})  conf={SRC_LM[n].confidence:.2f}"
          f"  [{SRC_LM[n].source}]")

plt.figure(figsize=(8, 8))
plt.imshow(flat(cc.draw_landmarks(SOURCE, SRC_LM)))
plt.title("REAL + landmarks"); plt.axis("off"); plt.show()


In [ ]:
#@title 9B. Landmarks da CHIBI (Run 003) { display-mode: "form" }
#@markdown O bloco `target` de `landmarks.yaml` vem **vazio de proposito**: a
#@markdown Run 003 nao esta versionada, entao preencher sem ver a imagem seria
#@markdown invencao. Confira o resultado automatico e, se estiver ruim, anote
#@markdown os valores no YAML e rode de novo.
TGT_LM = cc.derive_landmarks(BASE)
derivados_t = list(TGT_LM.names())
if OVR.get("target"):
    TGT_LM = cc.apply_overrides(TGT_LM, OVR["target"], source="config")

print(f"derivados do contorno ({len(derivados_t)}): {derivados_t}")
if not OVR.get("target"):
    print("\n[HUMAN REVIEW REQUIRED] Nenhum landmark de `target` no YAML.")
    print("A chibi tem cabeca proporcionalmente enorme e pescoco quase")
    print("inexistente; o contorno sozinho tende a nao achar cintura/quadril.")
for n in TGT_LM.names():
    nx, ny = TGT_LM.normalized(n)
    print(f"  {n:18} norm({nx:.3f}, {ny:.3f})  conf={TGT_LM[n].confidence:.2f}"
          f"  [{TGT_LM[n].source}]")

plt.figure(figsize=(8, 8))
plt.imshow(flat(cc.draw_landmarks(BASE, TGT_LM)))
plt.title("CHIBI (Run 003) + landmarks"); plt.axis("off"); plt.show()


In [ ]:
#@title 9C. Correspondencia e transformacao { display-mode: "form" }
#@markdown `auto`: TPS com >=4 pares, afim com 3, similaridade com 2.
#@markdown Force um tipo mais rigido se o TPS distorcer demais.
TIPO_TRANSFORMACAO = "auto"  #@param ["auto", "tps", "affine", "similarity"]
ANCORAS_DE_CAIXA = True  #@param {type:"boolean"}

try:
    CORR = cc.build_correspondence(SRC_LM, TGT_LM,
                                   kind=TIPO_TRANSFORMACAO,
                                   use_bbox_anchors=ANCORAS_DE_CAIXA)
except cc.CorrespondenceError as e:
    CORR = None
    print("FALHOU:", e)
    print("\nPares insuficientes. Preencha `target` em "
          f"characters/{CHARACTER_ID}/landmarks.yaml e rode de novo.")

if CORR:
    d = CORR.as_dict()
    print(f"transformacao : {d['kind']}")
    print(f"pares         : {d['n_pairs']} -> {d['pairs']}")
    print(f"confianca     : {d['confidence']:.3f}")
    print(f"so na REAL    : {d['diagnostics']['landmarks_source_only']}")
    print(f"so na CHIBI   : {d['diagnostics']['landmarks_target_only']}")
    print("\nconfianca e TRIAGEM, nao aprovacao. Quem aprova e voce, no olho.")
    plt.figure(figsize=(16, 8))
    plt.imshow(cc.draw_correspondence(SOURCE, BASE, CORR).convert("RGB"))
    plt.title(f"correspondencia · {d['kind']} · conf {d['confidence']:.3f}")
    plt.axis("off"); plt.show()


In [ ]:
#@title 9D. Mascaras transformadas para o espaco da CHIBI { display-mode: "form" }
#@markdown Depois do warp, `clip_to_target` prende a mascara dentro do corpo da
#@markdown chibi e fora das regioes protegidas DELA. E isso que sustenta
#@markdown `protected_overlap == 0` no espaco do alvo.
assert CORR is not None, "rode 9C com sucesso antes desta etapa"

tgt_shape = np.array(BASE).shape[:2]
tgt_subj = dt.subject_mask(BASE)

WARPED = cc.transform_masks(MASKS, CORR, tgt_shape)
prot_tgt = np.zeros(tgt_shape, bool)
for k, m in PROTECTED.items():
    prot_tgt |= cc.transform_mask(m, CORR, tgt_shape)

RAW = dict(WARPED)
WARPED = cc.clip_to_target(WARPED, tgt_subj, prot_tgt)

METRICS = cc.mask_metrics(MASKS, RAW, WARPED, tgt_subj, prot_tgt)

print("QUALIDADE DA CORRESPONDENCIA")
res = CORR.as_dict()["residuals"]
print(f"  transform type      : {CORR.kind}")
print(f"  correspondence points: {CORR.n_pairs}")
print(f"  rejected (duplicados): "
      f"{CORR.diagnostics.get('duplicate_points_dropped', 0)}")
print(f"  so na REAL (sem par) : "
      f"{len(CORR.diagnostics['landmarks_source_only'])}")
print(f"  residual rmse / max  : {res['rmse']} / {res['max']} px")
print(f"  confidence           : {CORR.confidence:.3f}")
if res["per_landmark"]:
    pior = max(res["per_landmark"].items(), key=lambda kv: kv[1])
    print(f"  pior landmark        : {pior[0]} ({pior[1]} px)")

print(f"\n{'peca':<14}{'REAL':>8}{'CHIBI':>8}{'ratio':>7}"
      f"{'clip':>8}{'fora':>7}{'prot':>6}")
for k in me.GARMENT_REGIONS:
    m = METRICS[k]
    print(f"{k:<14}{m['mask_area_source']:>8}{m['mask_area_target']:>8}"
          f"{(m['area_ratio'] or 0):>7.2f}{m['clipping_pixels']:>8}"
          f"{m['out_of_bounds_pixels']:>7}"
          f"{m['mask_protected_overlap_pixels']:>6}")

tot = METRICS["_total"]
print(f"\nmask_protected_overlap_pixels = "
      f"{tot['mask_protected_overlap_pixels']}"
      f"  {'OK' if tot['mask_protected_overlap_pixels'] == 0 else 'FALHOU'}")
print(f"out_of_bounds_pixels          = {tot['out_of_bounds_pixels']}")
print(f"clipping_pixels               = {tot['clipping_pixels']}")
print("\nclipping alto = correspondencia ruim, mesmo com overlap zero:")
print("o clip esconde o erro em vez de corrigi-lo. Olhe os overlays.")

fig, ax = plt.subplots(2, 3, figsize=(15, 11))
for a, name in zip(ax.ravel(), me.GARMENT_REGIONS):
    a.imshow(WARPED[name], cmap="gray")
    a.set_title(f"{name} -> CHIBI\n{int(WARPED[name].sum())} px")
    a.axis("off")
plt.suptitle("mascaras TRANSFORMADAS (espaco da Run 003)")
plt.tight_layout(); plt.show()


In [ ]:
#@title 9E. ORIGINAL vs TRANSFORMADA vs overlay no alvo { display-mode: "form" }
#@markdown A pergunta desta etapa: **a regiao da roupa da REAL foi parar na
#@markdown regiao equivalente da CHIBI?** A mascara precisa ficar dentro do
#@markdown personagem, respeitar as proporcoes da Run 003 e nao invadir
#@markdown rosto, cabelo nem chifres.
ALPHA_CORR = 0.55  #@param {type:"slider", min:0.1, max:0.9, step:0.05}

ov_src = me.overlay(SOURCE, MASKS, alpha=ALPHA_CORR)
ov_tgt = me.overlay(BASE, WARPED, alpha=ALPHA_CORR)

fig, ax = plt.subplots(1, 3, figsize=(18, 8))
for a, (img, t) in zip(ax, [(ov_src, "1. mascara ORIGINAL (REAL)"),
                            (ov_tgt, "2. mascara TRANSFORMADA (CHIBI)"),
                            (BASE, "3. Run 003 intacta")]):
    a.imshow(flat(img)); a.set_title(t); a.axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(2, 3, figsize=(16, 13))
for a, name in zip(ax.ravel(), me.GARMENT_REGIONS):
    a.imshow(flat(me.overlay(BASE, {name: WARPED[name]}, alpha=ALPHA_CORR)))
    a.set_title(f"{name} sobre Run 003")
    a.axis("off")
plt.suptitle("peca por peca — a mascara caiu no lugar certo?")
plt.tight_layout(); plt.show()

print("[HUMAN REVIEW REQUIRED] Aprovacao e sua.")
print("PARADA OBRIGATORIA: nada de warp da arte, composicao ou geracao final.")
print("A Run 003 nao foi alterada por nenhuma etapa deste notebook.")


In [ ]:
#@title 9. Metricas, hashes e ZIP da MASK REVIEW { display-mode: "form" }
import json, platform, shutil
import skimage

metrics_doc = {
    "stage": "mask_review",
    "generative": False,
    "character": CHARACTER_ID,
    "source": {"path": str(FULL_BODY),
               "sha256": __import__("hashlib").sha256(
                   FULL_BODY.read_bytes()).hexdigest()},
    "subject_bbox": list(REVIEW.subject_bbox),
    "protected_regions": {
        k: int(np.count_nonzero(v)) for k, v in PROTECTED.items()},
    "garment": REVIEW.metrics,
    "all_protected_overlap_zero": all(
        REVIEW.metrics[k]["protected_overlap"] == 0 for k in me.GARMENT_REGIONS),
    "libraries": {"numpy": np.__version__, "scikit-image": skimage.__version__,
                  "python": platform.python_version()},
    "licenses": {"numpy": "BSD-3-Clause", "pillow": "MIT-CMU",
                 "scikit-image": "BSD-3-Clause", "scipy": "BSD-3-Clause"},
    "parameters": REVIEW.params.values,
    "parameter_provenance": REVIEW.params.provenance,
    "profile": {
        "character_id": REVIEW.params.profile.character_id,
        "is_empty": REVIEW.params.profile.is_empty,
        "overridden": REVIEW.params.overridden,
        "path": str(REVIEW.params.profile.source_path or ""),
    },
    "approved": False,
    "human_review_required": [
        "[HUMAN REVIEW REQUIRED] Aprovar cada mascara visualmente antes de "
        "qualquer warp ou composicao."],
}
p = WORK / "mask_review.json"
p.write_text(json.dumps(metrics_doc, indent=2, ensure_ascii=False, default=str))
print(json.dumps(metrics_doc["garment"], indent=2, ensure_ascii=False, default=str))
print("\nsalvo:", p.relative_to(ROOT))

zb = WORK.parent / "mask_review_run_003"
shutil.make_archive(str(zb), "zip", WORK)
print("ZIP:", zb.with_suffix(".zip"))
try:
    from google.colab import files
    files.download(str(zb.with_suffix(".zip")))
except Exception:
    pass


In [ ]:
#@title 10. PORTAO — warp/composicao bloqueados { display-mode: "form" }
#@markdown A proxima etapa so roda apos SUA aprovacao visual das mascaras.
#@markdown Nao marque isto sem ter olhado cada overlay acima.
MASCARAS_APROVADAS = False  #@param {type:"boolean"}

objetivo_ok = all(REVIEW.metrics[k]["protected_overlap"] == 0
                  for k in me.GARMENT_REGIONS)
print("verificacoes objetivas (overlap zero, nao-vazias):", objetivo_ok)
print("aprovacao humana:", MASCARAS_APROVADAS)

if not MASCARAS_APROVADAS:
    print("""
=========================================================
MASK REVIEW — AGUARDANDO AVALIACAO HUMANA
=========================================================
Warp e composicao NAO serao executados.

Confira em cada overlay:
  [ ] torso cobre a gola/peca central, sem pele
  [ ] mangas cobrem as mangas sino, sem cabelo
  [ ] capa esquerda / direita cobrem o manto ate a barra
  [ ] ornamentos cobrem o ouro (ombreiras, coxas, cinto)
  [ ] inferiores cobrem o calcado
  [ ] nada toca rosto, olhos, boca, cabelo, chifres, maos, pernas

Se alguma estiver errada, relate QUAL peca e o que sobra ou falta.
""")
else:
    print("\nAprovado. A etapa de warp/composicao sera habilitada "
          "em uma proxima entrega, apos seu aval.")
